# 🛒 E-Commerce Sales Analysis Dashboard
### Project 1 — End-to-End Sales Performance Analysis
**Dataset:** 9,994 transactions | **Period:** 2011–2014 | **Region:** United States

---
**Business Problem:** Management lacks centralized visibility into e-commerce sales performance, making it difficult to identify underperforming regions/products or track critical Year-over-Year (YOY) profit and order growth metrics.

**Project Goal:** Perform end-to-end analysis using Python to uncover actionable trends in revenue, seasonality, product performance, and regional distribution.

---


## 📦 1. Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# ── Aesthetics ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.family': 'DejaVu Sans',
})

COLORS = {
    'primary':   '#2E86AB',
    'secondary': '#E84855',
    'accent':    '#F4A261',
    'success':   '#2DC653',
    'purple':    '#7B2D8B',
    'palette':   ['#2E86AB','#E84855','#F4A261','#2DC653','#7B2D8B',
                  '#F72585','#4361EE','#4CC9F0'],
}

df = pd.read_excel('Project_1Ecommerce_Sales_Analysis.xlsx', sheet_name='Data')
df['Month'] = df['Order Date'].dt.month
df['Month_Name'] = df['Order Date'].dt.strftime('%b')
print(f"✅ Data loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"   Years covered : {sorted(df['Year'].unique())}")
print(f"   Categories    : {list(df['Category'].unique())}")
print(f"   Regions       : {list(df['Region'].unique())}")
print(f"   Segments      : {list(df['Segment'].unique())}")
df.head(3)


Matplotlib is building the font cache; this may take a moment.


FileNotFoundError: [Errno 2] No such file or directory: 'Project_1Ecommerce_Sales_Analysis.xlsx'

## 🔍 2. Exploratory Data Analysis (EDA)

In [ ]:
print("=" * 55)
print("   DATASET OVERVIEW")
print("=" * 55)
print(f"  Total Rows       : {len(df):,}")
print(f"  Total Columns    : {df.shape[1]}")
print(f"  Missing Values   : {df.isnull().sum().sum()}")
print(f"  Duplicate Rows   : {df.duplicated().sum()}")
print()
print("  COLUMN DATA TYPES:")
print(df.dtypes.to_string())
print()
print("NUMERIC SUMMARY:")
df[['Sales','Quantity','Discount','Profit']].describe().round(2)


## 📊 3. Overall KPI Summary (All Years)
> Addresses: *"What is our total annual revenue, net profit, and overall profit margin?"*

In [ ]:
total_sales   = df['Sales'].sum()
total_profit  = df['Profit'].sum()
total_orders  = df['Order ID'].nunique()
total_qty     = df['Quantity'].sum()
profit_margin = (total_profit / total_sales) * 100

print("=" * 55)
print("   OVERALL BUSINESS KPIs (2011 – 2014)")
print("=" * 55)
print(f"  💰 Total Revenue      : ${total_sales:>12,.2f}")
print(f"  📈 Total Profit       : ${total_profit:>12,.2f}")
print(f"  📦 Total Orders       : {total_orders:>12,}")
print(f"  🛒 Total Quantity     : {total_qty:>12,}")
print(f"  📊 Profit Margin      : {profit_margin:>11.2f}%")
print("=" * 55)

fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
fig.suptitle('Overall KPI Dashboard (2011–2014)', fontsize=14, fontweight='bold', y=1.02)

kpis = [
    ("Total Revenue",    f"${total_sales/1e6:.2f}M",   COLORS['primary']),
    ("Total Profit",     f"${total_profit/1e3:.1f}K",  COLORS['success']),
    ("Profit Margin",    f"{profit_margin:.1f}%",       COLORS['accent']),
    ("Total Orders",     f"{total_orders:,}",           COLORS['secondary']),
    ("Total Qty Sold",   f"{total_qty:,}",              COLORS['purple']),
]
for ax, (label, val, color) in zip(axes, kpis):
    ax.set_facecolor(color + '22')
    ax.text(0.5, 0.6, val,  ha='center', va='center', fontsize=22, fontweight='bold', color=color, transform=ax.transAxes)
    ax.text(0.5, 0.25, label, ha='center', va='center', fontsize=10, color='#333', transform=ax.transAxes)
    for s in ax.spines.values(): s.set_visible(False)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig('kpi_dashboard.png', bbox_inches='tight')
plt.show()
print("\nFigure saved: kpi_dashboard.png")


## 📈 4. Year-over-Year (YOY) Growth
> Addresses: *"How do sales and profit compare year-over-year?"*

In [ ]:
yoy = df.groupby('Year').agg(
    Sales  = ('Sales','sum'),
    Profit = ('Profit','sum'),
    Orders = ('Order ID','nunique')
).reset_index()

yoy['Sales_YOY%']  = yoy['Sales'].pct_change()  * 100
yoy['Profit_YOY%'] = yoy['Profit'].pct_change() * 100
yoy['Orders_YOY%'] = yoy['Orders'].pct_change() * 100

print("YOY PERFORMANCE TABLE:")
print(yoy.to_string(index=False, float_format='{:.2f}'.format))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Year-over-Year Performance (2011–2014)', fontsize=14, fontweight='bold')

metrics = [
    ('Sales',  'Sales (USD)',   COLORS['primary']),
    ('Profit', 'Profit (USD)',  COLORS['success']),
    ('Orders', 'No. of Orders', COLORS['secondary']),
]
for ax, (col, ylabel, color) in zip(axes, metrics):
    bars = ax.bar(yoy['Year'], yoy[col], color=color, alpha=0.85, edgecolor='white', width=0.6)
    ax.set_title(f'Annual {col}', fontweight='bold')
    ax.set_xlabel('Year'); ax.set_ylabel(ylabel)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f'${x/1e3:.0f}K' if col != 'Orders' else f'{x:,.0f}'))
    for bar in bars:
        h = bar.get_height()
        label = f'${h/1e3:.1f}K' if col != 'Orders' else f'{h:,}'
        ax.text(bar.get_x() + bar.get_width()/2, h * 1.01, label,
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    # YOY % annotation
    for i, (_, row) in enumerate(yoy.iterrows()):
        if i > 0:
            pct = row[f'{col}_YOY%']
            sign = '+' if pct > 0 else ''
            ax.text(row['Year'], yoy[col].max() * 0.55,
                    f"{sign}{pct:.1f}%", ha='center', fontsize=9,
                    color=COLORS['success'] if pct > 0 else COLORS['secondary'])

plt.tight_layout()
plt.savefig('yoy_performance.png', bbox_inches='tight')
plt.show()
print("Figure saved: yoy_performance.png")


## 📅 5. Monthly Sales & Profit Seasonality
> Addresses: *"How do sales/profit fluctuate month-by-month? Are there months where profit dips despite high sales?"*

In [ ]:
month_order = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
monthly = df.groupby(['Month','Month_Name']).agg(
    Sales  = ('Sales','sum'),
    Profit = ('Profit','sum'),
    Orders = ('Order ID','nunique')
).reset_index().sort_values('Month')

fig, ax1 = plt.subplots(figsize=(14, 5))
fig.suptitle('Monthly Sales vs Orders (Combo Chart — Q1: Sales & Order Volume)', fontsize=13, fontweight='bold')

bars = ax1.bar(monthly['Month_Name'], monthly['Sales'], color=COLORS['primary'],
               alpha=0.75, label='Sales (USD)', zorder=2)
ax1.set_xlabel('Month'); ax1.set_ylabel('Sales (USD)', color=COLORS['primary'])
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax1.tick_params(axis='y', labelcolor=COLORS['primary'])

ax2 = ax1.twinx()
ax2.plot(monthly['Month_Name'], monthly['Orders'], color=COLORS['secondary'],
         marker='o', linewidth=2.5, label='Orders', zorder=3)
ax2.set_ylabel('Number of Orders', color=COLORS['secondary'])
ax2.tick_params(axis='y', labelcolor=COLORS['secondary'])
ax2.spines['right'].set_visible(True)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('monthly_sales_orders.png', bbox_inches='tight')
plt.show()

# Profit seasonality
fig, ax = plt.subplots(figsize=(14, 4))
fig.suptitle('Monthly Profit Trend (Q2: Peak Performance Month)', fontsize=13, fontweight='bold')
colors = [COLORS['success'] if p > 0 else COLORS['secondary'] for p in monthly['Profit']]
ax.bar(monthly['Month_Name'], monthly['Profit'], color=colors, alpha=0.85)
ax.plot(monthly['Month_Name'], monthly['Profit'], color='#333', marker='D',
        linewidth=2, markersize=6, zorder=3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax.set_xlabel('Month'); ax.set_ylabel('Profit (USD)')
ax.axhline(monthly['Profit'].mean(), linestyle='--', color='gray', alpha=0.7, label='Avg Profit')
ax.legend()
plt.tight_layout()
plt.savefig('monthly_profit.png', bbox_inches='tight')
plt.show()
print("Figures saved.")

peak_s = monthly.loc[monthly['Sales'].idxmax(), 'Month_Name']
peak_p = monthly.loc[monthly['Profit'].idxmax(), 'Month_Name']
low_p  = monthly.loc[monthly['Profit'].idxmin(), 'Month_Name']
print(f"\n📌 Peak Sales Month  : {peak_s}")
print(f"📌 Peak Profit Month : {peak_p}")
print(f"⚠️  Lowest Profit    : {low_p}")


## 🗂️ 6. Product Category Analysis
> Addresses: *"Which category commands the highest share of sales and profit?"*

In [ ]:
cat = df.groupby('Category').agg(
    Sales  = ('Sales','sum'),
    Profit = ('Profit','sum'),
    Orders = ('Order ID','nunique')
).reset_index()
cat['Margin%'] = (cat['Profit'] / cat['Sales'] * 100).round(2)
cat['Sales_Share%']  = (cat['Sales']  / cat['Sales'].sum()  * 100).round(2)
cat['Profit_Share%'] = (cat['Profit'] / cat['Profit'].sum() * 100).round(2)

print("CATEGORY PERFORMANCE TABLE:")
print(cat.to_string(index=False, float_format='{:.2f}'.format))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Q3: Category Sales & Profit Share', fontsize=13, fontweight='bold')

# Donut – Sales
wedges, texts, autotexts = axes[0].pie(
    cat['Sales'], labels=cat['Category'], autopct='%1.1f%%',
    colors=COLORS['palette'][:3], startangle=90,
    wedgeprops={'width':0.55, 'edgecolor':'white','linewidth':2})
axes[0].set_title('Sales Share by Category')
for at in autotexts: at.set_fontsize(11); at.set_fontweight('bold')

# Grouped Bar – Sales vs Profit
x = np.arange(len(cat))
w = 0.35
axes[1].bar(x - w/2, cat['Sales'],  width=w, label='Sales',  color=COLORS['primary'],  alpha=0.85)
axes[1].bar(x + w/2, cat['Profit'], width=w, label='Profit', color=COLORS['success'], alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(cat['Category'])
axes[1].set_title('Sales vs Profit by Category')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
axes[1].legend()

plt.tight_layout()
plt.savefig('category_analysis.png', bbox_inches='tight')
plt.show()
print("Figure saved: category_analysis.png")


## 🏆 7. Top 5 Sub-Categories by Sales
> Addresses: *"What are the top 5 subcategories driving the highest sales volume?"*

In [2]:
subcat = df.groupby('Sub-Category').agg(
    Sales  = ('Sales','sum'),
    Profit = ('Profit','sum')
).reset_index().sort_values('Sales', ascending=False)

top5  = subcat.head(5)
all5  = subcat.sort_values('Sales', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Q3 (cont.): Sub-Category Performance', fontsize=13, fontweight='bold')

# Top 5 horizontal bar
bar_colors = [COLORS['primary'] if i < 5 else '#aaa' for i in range(len(top5))]
axes[0].barh(top5['Sub-Category'], top5['Sales'], color=bar_colors, edgecolor='white')
axes[0].set_title('Top 5 Sub-Categories by Sales')
axes[0].set_xlabel('Sales (USD)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
for i, (_, row) in enumerate(top5.iterrows()):
    axes[0].text(row['Sales'] * 1.01, i, f"${row['Sales']/1e3:.1f}K",
                 va='center', fontsize=9, fontweight='bold')

# All sub-cats profit – highlight positive vs negative
colors_p = [COLORS['success'] if p > 0 else COLORS['secondary'] for p in all5['Profit']]
axes[1].barh(all5['Sub-Category'], all5['Profit'], color=colors_p, edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Profit by Sub-Category (All)')
axes[1].set_xlabel('Profit (USD)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

pos_patch = mpatches.Patch(color=COLORS['success'], label='Profitable')
neg_patch = mpatches.Patch(color=COLORS['secondary'], label='Loss-making')
axes[1].legend(handles=[pos_patch, neg_patch])

plt.tight_layout()
plt.savefig('subcategory_analysis.png', bbox_inches='tight')
plt.show()
print("Figure saved: subcategory_analysis.png")
print("\nTop 5 Sub-Categories:")
print(top5[['Sub-Category','Sales','Profit']].to_string(index=False, float_format='{:.2f}'.format))


NameError: name 'df' is not defined

## 🗺️ 8. Geographic Sales Distribution
> Addresses: *"Which states generate the highest sales revenue?"*

In [ ]:
state_sales = df.groupby('State').agg(
    Sales  = ('Sales','sum'),
    Profit = ('Profit','sum'),
    Orders = ('Order ID','nunique')
).reset_index().sort_values('Sales', ascending=False)

print("TOP 15 STATES BY SALES:")
print(state_sales.head(15).to_string(index=False, float_format='{:.2f}'.format))

region_sales = df.groupby('Region').agg(
    Sales  = ('Sales','sum'),
    Profit = ('Profit','sum')
).reset_index().sort_values('Sales', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Q5: Geographic Sales Distribution', fontsize=13, fontweight='bold')

# Top 10 States
top10 = state_sales.head(10).sort_values('Sales')
bar_cols = [COLORS['primary'] if i >= 7 else COLORS['accent'] for i in range(10)]
axes[0].barh(top10['State'], top10['Sales'], color=bar_cols, edgecolor='white', alpha=0.9)
axes[0].set_title('Top 10 States by Revenue')
axes[0].set_xlabel('Sales (USD)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
for _, row in top10.iterrows():
    axes[0].text(row['Sales'] * 1.01, row['State'], f"${row['Sales']/1e3:.0f}K",
                 va='center', fontsize=8.5)

# Region pie
wedges, texts, autotexts = axes[1].pie(
    region_sales['Sales'], labels=region_sales['Region'],
    autopct='%1.1f%%', colors=COLORS['palette'][:4],
    startangle=90, wedgeprops={'width':0.55, 'edgecolor':'white','linewidth':2})
axes[1].set_title('Sales Share by Region')
for at in autotexts: at.set_fontsize(11); at.set_fontweight('bold')

plt.tight_layout()
plt.savefig('geographic_analysis.png', bbox_inches='tight')
plt.show()
print("Figure saved: geographic_analysis.png")


## 👥 9. Customer Segment Analysis
> Addresses: *"How do metrics look by Market Segment (Consumer vs Corporate vs Home Office)?"*

In [ ]:
seg = df.groupby('Segment').agg(
    Sales  = ('Sales','sum'),
    Profit = ('Profit','sum'),
    Orders = ('Order ID','nunique')
).reset_index()
seg['Margin%'] = (seg['Profit'] / seg['Sales'] * 100).round(2)
print("SEGMENT PERFORMANCE:")
print(seg.to_string(index=False, float_format='{:.2f}'.format))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Q6: Customer Segment Performance', fontsize=13, fontweight='bold')

# Donut – Sales by Segment
wedges, texts, autotexts = axes[0].pie(
    seg['Sales'], labels=seg['Segment'], autopct='%1.1f%%',
    colors=COLORS['palette'][:3], startangle=90,
    wedgeprops={'width':0.55, 'edgecolor':'white','linewidth':2})
axes[0].set_title('Sales by Segment')
for at in autotexts: at.set_fontsize(11); at.set_fontweight('bold')

# Bar – Profit by Segment
axes[1].bar(seg['Segment'], seg['Profit'], color=COLORS['palette'][:3],
            edgecolor='white', alpha=0.9)
axes[1].set_title('Profit by Segment')
axes[1].set_ylabel('Profit (USD)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
for i, row in seg.iterrows():
    axes[1].text(i, row['Profit'] * 1.02, f"${row['Profit']/1e3:.1f}K",
                 ha='center', fontsize=9, fontweight='bold')

# Segment × Category clustered bar
seg_cat = df.groupby(['Segment','Category'])['Sales'].sum().unstack()
seg_cat.plot(kind='bar', ax=axes[2], color=COLORS['palette'][:3], edgecolor='white',
             alpha=0.9, rot=0)
axes[2].set_title('Sales by Segment × Category')
axes[2].set_ylabel('Sales (USD)')
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
axes[2].legend(title='Category', fontsize=8)

plt.tight_layout()
plt.savefig('segment_analysis.png', bbox_inches='tight')
plt.show()
print("Figure saved: segment_analysis.png")


## 🚚 10. Order Status / Ship Mode Analysis
> Addresses: *"Q4: Operational Order Status / Sales Channel Performance"*

In [ ]:
ship = df.groupby('Ship Mode').agg(
    Sales  = ('Sales','sum'),
    Orders = ('Order ID','nunique')
).reset_index().sort_values('Sales', ascending=False)
ship['Sales_Share%'] = (ship['Sales'] / ship['Sales'].sum() * 100).round(1)
print("SHIP MODE PERFORMANCE:")
print(ship.to_string(index=False, float_format='{:.2f}'.format))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Q4 & Q7: Ship Mode & Order Distribution', fontsize=13, fontweight='bold')

# Exploded Pie
explode = [0.05] * len(ship)
wedges, texts, autotexts = axes[0].pie(
    ship['Sales'], labels=ship['Ship Mode'], autopct='%1.1f%%',
    colors=COLORS['palette'][:4], startangle=90, explode=explode,
    wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title('Sales by Ship Mode (Exploded Pie)')
for at in autotexts: at.set_fontsize(10); at.set_fontweight('bold')

# Bar
axes[1].bar(ship['Ship Mode'], ship['Orders'], color=COLORS['palette'][:4],
            edgecolor='white', alpha=0.9)
axes[1].set_title('Number of Orders by Ship Mode')
axes[1].set_ylabel('Number of Orders')
for i, row in ship.reset_index().iterrows():
    axes[1].text(i, row['Orders'] * 1.01, f"{row['Orders']:,}",
                 ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('shipmode_analysis.png', bbox_inches='tight')
plt.show()
print("Figure saved: shipmode_analysis.png")


## 🔍 11. Slicer-Style Filtered Analysis (Year × Region × Segment)
> Addresses: *"How do performance metrics look when isolated by Year, Region, or Segment?"*

In [ ]:
def dashboard_slice(year=None, region=None, segment=None):
    """Filter data and print a KPI summary — mimics an Excel Slicer."""    data = df.copy()
    if year:    data = data[data['Year']    == year]
    if region:  data = data[data['Region']  == region]
    if segment: data = data[data['Segment'] == segment]
    
    sales  = data['Sales'].sum()
    profit = data['Profit'].sum()
    orders = data['Order ID'].nunique()
    margin = (profit / sales * 100) if sales > 0 else 0
    top_cat = data.groupby('Category')['Sales'].sum().idxmax()
    
    label_parts = []
    if year:    label_parts.append(f"Year={year}")
    if region:  label_parts.append(f"Region={region}")
    if segment: label_parts.append(f"Segment={segment}")
    label = ' | '.join(label_parts) if label_parts else 'ALL DATA'
    
    print(f"  Filter : {label}")
    print(f"  Sales  : ${sales:>12,.2f}")
    print(f"  Profit : ${profit:>12,.2f}  (Margin: {margin:.1f}%)")
    print(f"  Orders : {orders:>12,}")
    print(f"  Top Cat: {top_cat}")
    print("-" * 48)

print("=" * 48)
print("   INTERACTIVE SLICER ANALYSIS")
print("=" * 48)
dashboard_slice()
dashboard_slice(year=2014)
dashboard_slice(region='West')
dashboard_slice(segment='Consumer')
dashboard_slice(year=2014, region='West')
dashboard_slice(year=2014, region='East', segment='Corporate')


## 💡 12. Key Business Insights & Recommendations

| # | Insight | Recommendation |
|---|---------|----------------|
| 1 | **Revenue grew 51.6%** from 2011 ($484K) to 2014 ($734K) — strong upward trend | Continue investing in high-growth product lines; set 2015 target at $900K |
| 2 | **November & December** are peak months; **January** is the weakest | Launch pre-holiday promotions in October; add January flash sales |
| 3 | **Technology** leads in profit ($145K, 17.4% margin); **Furniture** lags (2.5% margin) | Review Furniture pricing and high-discount items like Tables |
| 4 | **California, New York & Texas** account for ~41% of total revenue | Focus marketing budgets on these 3 states; expand into Ohio & Michigan |
| 5 | **Consumer segment** drives 50.6% of sales; **Home Office** is smallest but fastest growing | Introduce Home Office bundles; run targeted corporate loyalty programs |
| 6 | **Standard Class shipping** dominates (59%) | Consider same-day incentives for high-value corporate orders |
| 7 | Sub-categories **Tables** and **Bookcases** show losses | Reassess discount policies; consider removal from catalog or price correction |
| 8 | **Profit margin is 12.46%** — healthy but with room to improve via discount controls | Cap discounts above 30%; audit rows where Discount ≥ 0.4 |

---
*Analysis performed using Python (pandas, matplotlib). Dataset: US E-Commerce 2011–2014 (9,994 transactions).*
